In [28]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain

load_dotenv()

True

In [8]:
loader = PyPDFLoader("tetum_cv.pdf")
docs = loader.load()
print(docs)

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2025-08-05T01:25:51+00:00', 'title': 'White and Dark Grey Minimalistic Profesional Cv Resume', 'moddate': '2025-08-05T01:25:50+00:00', 'keywords': 'DAGvKItwqCI,BAGP2GbFwzY,0', 'author': 'Juliao De Jesus Martins', 'source': 'tetum_cv.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='IT SUPPORT | WEB DEVELOPER\nJULIÃO DE\nJESUS MARTINS\n+670 74242390\njuliaodejesusmartins@gmail.com\nDom Aleixo, Comoro, 12\nde Outubro\nKONABA HAU\nHau Julião de Jesus Martins, profisionál IT Support motivadu ho\nintegridade, atu ajuda ema, solusiona problema, kontínua aprende\nno fó valór ba oportunidade.\nEXPERENSIAjuliao-martins.vercel.app\nIT Support iha Leader group responsavel ba iha unit hanesan Leader\nHypermarket Lda, Leader Mart Lda, Lisun Importacao no\nExportacao Lda, Cement Timor Sa, no Auto Timor-Leste Lda\nresponsavel ekipamentu Informatika no linha rede internet tenke\nfunsiona ho diak no seguru.

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(docs)
print(len(chunks))

1


In [32]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorestore.save_local("tetum_cv_index")

In [40]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

llm = ChatOpenAI(model="gpt-4")

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    memory=memory,
    retriever=vectorstore.as_retriever()
)

question = [
    "Who's Juliao de Jesus Martins?",
    # "How many total years of work experience does Julião de Jesus Martins have?",
]

for q in question:
    print("Question:", q)
    result = qa_chain.invoke({"question": q})
    print("Answer:", result["answer"])
    

Question: Who's Juliao de Jesus Martins?
Answer: Julião de Jesus Martins is a professional in IT Support who appears to be very motivated and has a commitment to helping others, solving problems, and continuous learning. He has experience as an IT Support at a Leader group where he was responsible for various units such as Leader Hypermarket Ltda, Leader Mart Ltda, Lisun Importacao no Exportacao Ltda, Cement Timor Sa, and Auto Timor-Leste Ltda. His responsibilities included ensuring the functioning and security of IT equipment and the internet network. He also has experience as a teacher at ETI Dili, where he taught the discipline of Multimedia and web programming using technologies such as HTML, CSS, JavaScript, PHP, and MySQL. He is also able to train new users and has knowledge in the field of Artificial Intelligence.
